## Assessment 2

Use dataset from Assessment 1 to train a machine learning model on historical records then appluy it to a simulated real-time stream. This assessment supports unit learning outcomes 3,4 and 5. 

### Before Part A
Data is split into training dataset, 70% of records, to train and evaluate the ML model and remaining 30% will be used for streaming dataset and will be replayed through Kfka in Part B.

In [4]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

#setup a spark session and load dataset
master = "local[*]"
app_name = "Uber Business Query DF"
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()

sc = spark.sparkContext

df = spark.read.csv("ncr_ride_bookings.csv",header=True)

In [5]:
train_df1, stream_df = df.randomSplit([0.7,0.3],seed = 42)

#to ensure a 70/30 split of data
print(train_df1.count())
print(train_df1.count()/150000)
print(stream_df.count())
print(stream_df.count()/150000)

#repeat of assessment one to ensure data type is correct
from pyspark.sql.functions import col,concat_ws
from pyspark.sql.types import DateType

train_df1 = train_df1.replace("null", None)
float_col = ["Avg VTAT","Avg CTAT","Ride Distance","Driver Ratings","Customer Rating","Booking Value"]
train_df1 = train_df1.select([col(c).cast("float") if c in float_col else col(c) for c in df.columns])

#remove all entries that do no have a driver rating number, which is the number the customer gives to the driver
#this is done after the initial split to reflect what real-time data would be, as the driver rating is optional for customers
train_df = train_df1.dropna(subset=["Driver Ratings"])
print(train_df.count())

105209
0.7013933333333333
44791
0.2986066666666667
65201


### Part A - The ML Pipeline

#### 1. The ML task selection

The ideal machine learning algorithm for the original business case of trying to predict the driver rating of their ride based on the record features would be regression. This is becuase the ratings are numerical in value, continuous after some data cleaning, and do have meaningful variance. The dataset included will only included records that do have a driver rating, which is much lower than the original dataset but still large enough to create a model with meaningful result. 

#### 2. Feature Enginerring

For this dataset, there are four general types of data that is included - categorical, binary, numerical and free text. Free text is seperated as it required further work to categorise this data, such as feature extraction and feature transformation. Binary is also removed, as it can easily be converted to full numerical with NULL coverted to 0. Booking ID and Customer ID are removed, as they are unique and identifying features that would not provide any predictable value. 

In [6]:
train_df.show(5)

+----------+--------+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+
|      Date|    Time|      Booking ID|Booking Status|     Customer ID|Vehicle Type|Pickup Location|Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|
+----------+--------+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-------------

In [7]:
#change the date and time into categorical values that would be impactful in prediction, specifically month, day and hour
#change hour, dayofweek and month into strings

from pyspark.sql.functions import hour, dayofweek, month

train_df = (train_df.withColumn("Hour", hour("Time").cast("string")).withColumn("DayOfWeek", dayofweek("Date").cast("string"))
.withColumn("Month", month("Date").cast("string")))
train_df.show(5)

+----------+--------+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+----+---------+-----+
|      Date|    Time|      Booking ID|Booking Status|     Customer ID|Vehicle Type|Pickup Location|Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|Hour|DayOfWeek|Month|
+----------+--------+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+---------------

In [8]:
train_df = train_df.drop("Date", "Time")
train_df.show(5)

+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+----+---------+-----+
|      Booking ID|Booking Status|     Customer ID|Vehicle Type|Pickup Location|Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|Hour|DayOfWeek|Month|
+----------------+--------------+----------------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------

The date and time were converted into hour, day of the week and month as this will be more relevant to the final model. Even though they are numerical values representing specific features, they will be treated at categories as they represent a specific time of day or year rather than a numerical number. 

In [9]:
#seperate numerically relevant columns from category columns as well as binary columns and free text
categoryInputCols = ["Booking Status", "Vehicle Type", "Pickup Location", "Drop Location","Payment Method", "Hour", "DayOfWeek", "Month"]
binaryInputCols = ["Cancelled Rides by Customer","Cancelled Rides by Driver", "Incomplete Rides"]
freetextInputCols = ["Reason for cancelling by Customer","Driver Cancellation Reason","Incomplete Rides Reason"]
numericInputCols = ["Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance", "Customer Rating"]
numericOutputCol = "Driver Ratings"

numericalCols = [numericOutputCol]+numericInputCols

In [10]:
train_df.select(binaryInputCols).printSchema()

root
 |-- Cancelled Rides by Customer: string (nullable = true)
 |-- Cancelled Rides by Driver: string (nullable = true)
 |-- Incomplete Rides: string (nullable = true)



In [11]:
#covert binary input cols into full numerical with Null = 0.0. since column is string, need to state that rest of values are numerical

from pyspark.sql.functions import when

for c in binaryInputCols:
    train_df = train_df.withColumn(c,when(col(c).isNull(), 0.0).otherwise(col(c).cast("double")))
train_df.show()

+----------------+--------------+----------------+------------+-------------------+--------------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+----+---------+-----+
|      Booking ID|Booking Status|     Customer ID|Vehicle Type|    Pickup Location|       Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|Hour|DayOfWeek|Month|
+----------------+--------------+----------------+------------+-------------------+--------------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+---------------

#### 2a. Investigation into Payment method

As part of feature extraction, I needed to investiagte paymend method and impact on the specific part of the dataset I was looking at. Due to that, I went back and investigated the impact of removing all null Driver Ratings from above and reassessed what columns I needed within my model. After the below investigation, the concluding assumption is that only rides that are complete allow the customer to give the driver a rating. A rating cannot be given if the booking is incomplete or cancelled, however if it is incomplete a payment is still taken and recorded. Based on this investigation, it was determined that Booking Status and any information relating to cancellation or incomplete would have no influence on the predictive model to determine Driver Ratings. Therefore these will be removed from the features used in model.

In [12]:
#Review null values in category columns to see if data is missing or important

#only looking at the non-altered categories 
null_categoryInputCols = ["Booking Status", "Vehicle Type", "Pickup Location", "Drop Location","Payment Method"]

from pyspark.sql.functions import col, sum
print("dataset before the removing of null driver ratings")
train_df1.select([sum(col(c).isNull().cast("int")).alias(c)for c in null_categoryInputCols]).show()
print("dataset after the removing of null driver ratings")
train_df.select([sum(col(c).isNull().cast("int")).alias(c)for c in null_categoryInputCols]).show()
print("streaming dataset with null")
stream_df.select([sum(col(c).isNull().cast("int")).alias(c)for c in null_categoryInputCols]).show()
print("total dataset with null")
df.select([sum(col(c).isNull().cast("int")).alias(c)for c in null_categoryInputCols]).show()

dataset before the removing of null driver ratings
+--------------+------------+---------------+-------------+--------------+
|Booking Status|Vehicle Type|Pickup Location|Drop Location|Payment Method|
+--------------+------------+---------------+-------------+--------------+
|             0|           0|              0|            0|         33638|
+--------------+------------+---------------+-------------+--------------+

dataset after the removing of null driver ratings
+--------------+------------+---------------+-------------+--------------+
|Booking Status|Vehicle Type|Pickup Location|Drop Location|Payment Method|
+--------------+------------+---------------+-------------+--------------+
|             0|           0|              0|            0|             0|
+--------------+------------+---------------+-------------+--------------+

streaming dataset with null
+--------------+------------+---------------+-------------+--------------+
|Booking Status|Vehicle Type|Pickup Location

In [13]:
from pyspark.sql.functions import col, count

df.groupBy("Booking Status").agg(
    sum(col("Driver Ratings").isNull().cast("int")).alias("Missing Driver Rating"),
    sum(col("Payment Method").isNull().cast("int")).alias("Missing Payment Method"),
    count("*").alias("Total")
).show()

+--------------------+---------------------+----------------------+-----+
|      Booking Status|Missing Driver Rating|Missing Payment Method|Total|
+--------------------+---------------------+----------------------+-----+
|           Completed|                    0|                     0|93000|
|     No Driver Found|                    0|                     0|10500|
| Cancelled by Driver|                    0|                     0|27000|
|Cancelled by Cust...|                    0|                     0|10500|
|          Incomplete|                    0|                     0| 9000|
+--------------------+---------------------+----------------------+-----+



#### 2b. Vector Assesmbler

In [14]:
#final set of columns that will be used in vector assessmbler and model generation
#seperate numerically relevant columns from category columns as well as binary columns and free text
categoryInputCols = ["Vehicle Type", "Pickup Location", "Drop Location","Payment Method","Hour","DayOfWeek","Month"]
numericInputCols = ["Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance", "Customer Rating"]
numericOutputCol = "Driver Ratings"

numericalCols = [numericOutputCol]+numericInputCols

In [15]:
from pyspark.ml.feature import StringIndexer

# Define output column names
outputCols = [f'{x}_index' for x in categoryInputCols]

print(categoryInputCols)
print(outputCols)

# Create index values for categorical input features
inputIndexer = StringIndexer(inputCols=categoryInputCols,outputCols=outputCols,handleInvalid="keep")

# Call the fit and transform() method to get the encoded results 
train_df_indexed = inputIndexer.fit(train_df).transform(train_df)
# Display the output, only the output columns
train_df_indexed.select(outputCols).show(10)

['Vehicle Type', 'Pickup Location', 'Drop Location', 'Payment Method', 'Hour', 'DayOfWeek', 'Month']
['Vehicle Type_index', 'Pickup Location_index', 'Drop Location_index', 'Payment Method_index', 'Hour_index', 'DayOfWeek_index', 'Month_index']
+------------------+---------------------+-------------------+--------------------+----------+---------------+-----------+
|Vehicle Type_index|Pickup Location_index|Drop Location_index|Payment Method_index|Hour_index|DayOfWeek_index|Month_index|
+------------------+---------------------+-------------------+--------------------+----------+---------------+-----------+
|               1.0|                104.0|              118.0|                 4.0|      18.0|            5.0|        0.0|
|               2.0|                 16.0|              120.0|                 0.0|      12.0|            5.0|        0.0|
|               0.0|                 95.0|               23.0|                 4.0|       8.0|            5.0|        0.0|
|               3.

With the first step in feature extraction, the StringIndexer is converting categories into numerical values. It uses the column names identified above and assigns each type in that column a numerical number. For example, with Vechile Type there are approximately 7 different types and therefore the index assigned is 0-6. However, for location there are a lot more types available and therefore the numerical range is a lot higher.  

In [16]:
from pyspark.ml.feature import OneHotEncoder

# Input columns are the indexed categorical input columns
inputCols_OHE = outputCols

# Define output columns for the one-hot encoded vectors
outputCols_OHE = [f'{x}_vec' for x in categoryInputCols]

#Define OneHotEncoder with the appropriate columns
encoder = OneHotEncoder(inputCols=inputCols_OHE,outputCols=outputCols_OHE)

# Call fit and transform to get the encoded results
model = encoder.fit(train_df_indexed)
train_df_encoded = model.transform(train_df_indexed)
# Display the output columns
train_df_encoded.select(outputCols_OHE).show(5)

+----------------+-------------------+-----------------+------------------+---------------+-------------+--------------+
|Vehicle Type_vec|Pickup Location_vec|Drop Location_vec|Payment Method_vec|       Hour_vec|DayOfWeek_vec|     Month_vec|
+----------------+-------------------+-----------------+------------------+---------------+-------------+--------------+
|   (7,[1],[1.0])|  (176,[104],[1.0])|(176,[118],[1.0])|     (5,[4],[1.0])|(24,[18],[1.0])|(7,[5],[1.0])|(12,[0],[1.0])|
|   (7,[2],[1.0])|   (176,[16],[1.0])|(176,[120],[1.0])|     (5,[0],[1.0])|(24,[12],[1.0])|(7,[5],[1.0])|(12,[0],[1.0])|
|   (7,[0],[1.0])|   (176,[95],[1.0])| (176,[23],[1.0])|     (5,[4],[1.0])| (24,[8],[1.0])|(7,[5],[1.0])|(12,[0],[1.0])|
|   (7,[3],[1.0])|  (176,[169],[1.0])| (176,[67],[1.0])|     (5,[2],[1.0])| (24,[3],[1.0])|(7,[5],[1.0])|(12,[0],[1.0])|
|   (7,[0],[1.0])|   (176,[96],[1.0])| (176,[70],[1.0])|     (5,[1],[1.0])| (24,[6],[1.0])|(7,[5],[1.0])|(12,[0],[1.0])|
+----------------+--------------

The second part requires coverting the numerical data of each of the features into a sparse vector that stores non-zero values and their indices. For each vector the first value is the number of types available in that specific feature, so for Vechicle Type the first value for all vectors in 7, then the indices of where the value is within the all values, and finally the value itself. Since each vector is a single, the value for all entries will be 1.0. 

In [17]:
from pyspark.ml.feature import VectorAssembler

# inputCols are all the encoded columns from OHE plus numerical columns
assemblerInputs = outputCols_OHE + numericInputCols

# Combine all input features into a single feature vector
assembler = VectorAssembler(inputCols=assemblerInputs,outputCol="features")

# Transform the encoded dataframe
train_df_final = assembler.transform(train_df_encoded)

# Display features and numerical output
train_df_final.select("features", numericOutputCol).show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|features                                                                                                                                                            |Driver Ratings|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|(412,[1,111,301,363,382,393,395,407,408,409,410,411],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.800000190734863,44.0,261.0,25.030000686645508,4.5])                             |4.2           |
|(412,[2,23,303,359,376,393,395,407,408,409,410,411],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,14.699999809265137,23.100000381469727,344.0,16.760000228881836,4.900000095367432]) |3.8           |
|(412,[0,102,206,363,372,393,395,407,408,409,410,411],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,11.5,16

The final dataset is created using the vector assesmbler, which calls the sparse vector created above and combines all the features into a single sparse vector for all the features. The sparse vector still have 3 sections but now there are 412 values for each data entry. The final part of the sparse vector shows that the first part are categorical values, since they have 1.0 as the entry, but then goes to numerical values. Finally each data entry is compared to the output column to be used in the model, Drivers Rating.

#### 3/4. Model Comparision and Evaluation

Several models are created and compared using RMSE, R2 and MAE suitable for a regression model. RMSE or root mean square error is the square of the differences between predicted and actual value. For an ideal model, this value should be as close to zero as possible. R-squared is the variance of the predicted value. Close to 1 R2 is the better the model to fitting the dataset. Finally mean absolute error or MAE is the absolute difference between predicted and true value. It is simialr to RMSE but this is a more direct way of interpreting the accuracy of the model and ideal value would be close to 0 as possible. Since the model is trying to predict Drivers Rating, which is between 1-5, this would give an immediate insight into the model and how well it fits to training data. 

In [18]:
#train_df is further split into 70/30 to allow models to be trained. This training dataset will be used for all models. 
train, test = train_df_final.randomSplit([0.7, 0.3], seed = 2020)
print("Training Dataset Count: " + str(train.count()))
print("Test Dataset Count: " + str(test.count()))

train.show(5)


26/10/03 09:48:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Training Dataset Count: 45555


Test Dataset Count: 19646
+----------------+--------------+----------------+-------------+---------------+----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+----+---------+-----+------------------+---------------------+-------------------+--------------------+----------+---------------+-----------+----------------+-------------------+-----------------+------------------+---------------+-------------+--------------+--------------------+
|      Booking ID|Booking Status|     Customer ID| Vehicle Type|Pickup Location|   Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|H

In [19]:
#creating a function that i can use for all evaluations of my regression models
from pyspark.ml.evaluation import RegressionEvaluator

def evaluate_regression(predictions, labelCol):

    rmse_evaluator = RegressionEvaluator(labelCol=labelCol,predictionCol="prediction",metricName="rmse")
    r2_evaluator = RegressionEvaluator(labelCol=labelCol,predictionCol="prediction",metricName="r2")
    mae_evaluator = RegressionEvaluator(labelCol=labelCol, predictionCol = "prediction",metricName= "mae")
    
    rmse = rmse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    
    print(f"RMSE: {rmse:.3f}")
    print(f"R²:   {r2:.3f}")
    print(f"MAE:   {mae:.3f}")
    
    return rmse, r2, mae

In [20]:
from pyspark.ml.regression import LinearRegression

# Create an initial model using the train set, using basic linear regression with no additional settings
lr = LinearRegression(featuresCol = "features", labelCol=numericOutputCol)
lrModel = lr.fit(train)

lrPredictions = lrModel.transform(test)
lrPredictions.select(numericOutputCol,'prediction').show()

26/10/03 09:48:43 WARN Instrumentation: [565921c8] regParam is zero, which might cause numerical instability and overfitting.
26/10/03 09:48:44 WARN Instrumentation: [565921c8] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


+--------------+-----------------+
|Driver Ratings|       prediction|
+--------------+-----------------+
|           4.4|4.149727015335577|
|           4.6|4.290834459055684|
|           4.3|4.265141597746981|
|           4.2|4.216547416156796|
|           4.1|4.212980762453869|
|           4.2|4.248500940760183|
|           4.6|4.229237735088293|
|           4.7| 4.24497800920897|
|           4.5|4.258383070388338|
|           3.2|4.201292830464332|
|           4.1|4.183222754261615|
|           4.4|4.249449889676875|
|           4.9| 4.18905258800651|
|           4.7|4.199448337906711|
|           4.5|4.323200864801352|
|           4.2|4.276633147727592|
|           5.0|4.233456181080399|
|           3.1|4.243305908534888|
|           4.1|4.296325497918988|
|           3.8|4.209563365401953|
+--------------+-----------------+
only showing top 20 rows


In [21]:
evaluate_regression(lrPredictions,numericOutputCol)

RMSE: 0.439
R²:   -0.009
MAE:   0.333


(0.43935041911630623, -0.00866754124798308, 0.3332435890324029)

In [22]:
#introducing regularisation to control the fit of the model
lr_l2 = LinearRegression(featuresCol="features",labelCol=numericOutputCol,regParam=0.1,elasticNetParam=0.0)
lrl2_model = lr_l2.fit(train)
lrl2_Predictions = lrl2_model.transform(test)
evaluate_regression(lrl2_Predictions,numericOutputCol)


RMSE: 0.439
R²:   -0.006
MAE:   0.332


(0.4387037829129314, -0.005700611268282918, 0.332432740220418)

In [23]:

print(len(train.select("features").first()["features"]))

412


#### 3/4a. Removal of location feature

Based on the large number of features within the df_final dataset, mainly due to the large number of locations possible in the pickup and drop-off location, a seperate dataset will be created without those columns to see if this could create a better suited model.

In [24]:
categoryInputCols_nolocation = ["Vehicle Type","Payment Method","Hour","DayOfWeek","Month"]

# Define output column names
outputCols = [f'{x}_index' for x in categoryInputCols_nolocation]

# Create index values for categorical input features
inputIndexer1 = StringIndexer(inputCols=categoryInputCols_nolocation,outputCols=outputCols,handleInvalid="keep") 
train_df_indexed_noloc = inputIndexer1.fit(train_df).transform(train_df)

# Input columns are the indexed categorical input columns
inputCols_OHE = outputCols
outputCols_OHE = [f'{x}_vec' for x in categoryInputCols_nolocation]

#Define OneHotEncoder with the appropriate columns
encoder1 = OneHotEncoder(inputCols=inputCols_OHE,outputCols=outputCols_OHE)
model1 = encoder1.fit(train_df_indexed_noloc)
train_df_encoded1 = model1.transform(train_df_indexed_noloc)

# inputCols are all the encoded columns from OHE plus numerical columns
assemblerInputs1 = outputCols_OHE + numericInputCols
assembler1 = VectorAssembler(inputCols=assemblerInputs1,outputCol="features")
train_df_noloc= assembler1.transform(train_df_encoded1)

#number of features within the new dataset
print(len(train_df_noloc.select("features").first()["features"]))

train_noloc, test_noloc = train_df_noloc.randomSplit([0.7, 0.3], seed = 2020)
print("No location training Dataset Count: " + str(train_noloc.count()))
print("No location test Dataset Count: " + str(test_noloc.count()))

60
No location training Dataset Count: 45555
No location test Dataset Count: 19646


In [25]:
#repeat of linear regression with no location dataset
lr2 = LinearRegression(featuresCol="features",labelCol=numericOutputCol)
lr2_model = lr2.fit(train_noloc)
lr2_Predictions = lr2_model.transform(test_noloc)
evaluate_regression(lr2_Predictions,numericOutputCol)

26/10/03 09:48:51 WARN Instrumentation: [8ce39f11] regParam is zero, which might cause numerical instability and overfitting.
26/10/03 09:48:51 WARN Instrumentation: [8ce39f11] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


RMSE: 0.438
R²:   -0.001
MAE:   0.331


(0.43765859970665405, -0.0009142871363132521, 0.33136585385446443)

In [26]:
#repeat with regularisation parameter
lr2_l2 = LinearRegression(featuresCol="features",labelCol=numericOutputCol,regParam=0.0001,elasticNetParam=0.0)
lr2_l2_model = lr2_l2.fit(train_noloc)
lr2_l2_Predictions = lr2_l2_model.transform(test_noloc)
evaluate_regression(lr2_l2_Predictions,numericOutputCol)

RMSE: 0.438
R²:   -0.001
MAE:   0.331


(0.4376585104360168, -0.000913878816919178, 0.3313658350899033)

In [27]:
#trying a decision tree model to see if that improves the fitting
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(featuresCol="features",labelCol=numericOutputCol,seed=42)

# Train model with no location data
dt_model = dt.fit(train_noloc)
dt_predictions = dt_model.transform(test_noloc)

dt_results = evaluate_regression(dt_predictions,numericOutputCol)

RMSE: 0.438
R²:   -0.004
MAE:   0.332


In [28]:
dt1 = DecisionTreeRegressor(featuresCol="features",labelCol=numericOutputCol,maxDepth=5,minInstancesPerNode=10,seed=42)

# Train model with no location data
dt1_model = dt1.fit(train_noloc)
dt1_predictions = dt1_model.transform(test_noloc)

dt1_results = evaluate_regression(dt1_predictions,numericOutputCol)

RMSE: 0.438
R²:   -0.004
MAE:   0.332


In [29]:
dt2 = DecisionTreeRegressor(featuresCol="features",labelCol=numericOutputCol,maxDepth=15,minInstancesPerNode=5,seed=42)

# Train model with no location data
dt2_model = dt2.fit(train_noloc)
dt2_predictions = dt2_model.transform(test_noloc)

dt2_results = evaluate_regression(dt2_predictions,numericOutputCol)

RMSE: 0.465
R²:   -0.132
MAE:   0.357


In [30]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(featuresCol="features",labelCol=numericOutputCol,seed=42)

rf_model = rf.fit(train_noloc)
rf_predictions = rf_model.transform(test_noloc)

rf_results = evaluate_regression(rf_predictions,numericOutputCol)

RMSE: 0.438
R²:   -0.001
MAE:   0.331


In [31]:
rf1 = RandomForestRegressor(featuresCol="features",labelCol=numericOutputCol,numTrees=50,maxDepth=15,minInstancesPerNode=5,
    featureSubsetStrategy="sqrt",seed=42)

rf1_model = rf1.fit(train_noloc)
rf1_predictions = rf1_model.transform(test_noloc)

rf1_results = evaluate_regression(rf1_predictions,numericOutputCol)

26/10/03 09:49:04 WARN DAGScheduler: Broadcasting large task binary with size 1128.5 KiB
26/10/03 09:49:04 WARN DAGScheduler: Broadcasting large task binary with size 1759.5 KiB
26/10/03 09:49:04 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/10/03 09:49:05 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB
26/10/03 09:49:05 WARN DAGScheduler: Broadcasting large task binary with size 5.2 MiB
26/10/03 09:49:06 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/10/03 09:49:07 WARN DAGScheduler: Broadcasting large task binary with size 1119.0 KiB
26/10/03 09:49:07 WARN DAGScheduler: Broadcasting large task binary with size 9.3 MiB
26/10/03 09:49:07 WARN DAGScheduler: Broadcasting large task binary with size 1345.1 KiB
26/10/03 09:49:08 WARN DAGScheduler: Broadcasting large task binary with size 12.0 MiB
26/10/03 09:49:09 WARN DAGScheduler: Broadcasting large task binary with size 1581.0 KiB
26/10/03 09:49:09 WARN DAGScheduler: B

RMSE: 0.438
R²:   -0.004
MAE:   0.332


In [32]:
rf2 = RandomForestRegressor(featuresCol="features",labelCol=numericOutputCol,numTrees=130,maxDepth=10,minInstancesPerNode=5,
    featureSubsetStrategy="sqrt",seed=42)

rf2_model = rf2.fit(train_noloc)
rf2_predictions = rf2_model.transform(test_noloc)

rf2_results = evaluate_regression(rf2_predictions,numericOutputCol)

26/10/03 09:49:13 WARN DAGScheduler: Broadcasting large task binary with size 1524.7 KiB
26/10/03 09:49:14 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/10/03 09:49:15 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
26/10/03 09:49:16 WARN DAGScheduler: Broadcasting large task binary with size 6.2 MiB
26/10/03 09:49:16 WARN DAGScheduler: Broadcasting large task binary with size 1278.5 KiB
26/10/03 09:49:17 WARN DAGScheduler: Broadcasting large task binary with size 9.2 MiB
26/10/03 09:49:18 WARN DAGScheduler: Broadcasting large task binary with size 1750.3 KiB


RMSE: 0.438
R²:   -0.001
MAE:   0.331


In [33]:
rf3 = RandomForestRegressor(featuresCol="features",labelCol=numericOutputCol,numTrees=20,maxDepth=5,minInstancesPerNode=10,
    featureSubsetStrategy="sqrt",seed=42)

rf3_model = rf3.fit(train_noloc)
rf3_predictions = rf3_model.transform(test_noloc)

rf3_results = evaluate_regression(rf3_predictions,numericOutputCol)

RMSE: 0.438
R²:   -0.000
MAE:   0.331


#### 3/4b. Investigation into important features

As most of these models have very similar metrics and none showing a good fitting model, an investigation into the most influential features will be done. After, the dataset may be reduced further to have only these features included. 

In [34]:
#translating the features from the spare vector back into understandable categories
metadata = train_df_noloc.schema["features"].metadata
attrs = metadata["ml_attr"]["attrs"]
feature_names = []

for attr_type in attrs:
    for attr in attrs[attr_type]:
        feature_names.append((attr["idx"], attr["name"]))

# Sort by feature index
feature_names = [name for idx, name in sorted(feature_names)]
print(feature_names)

['Vehicle Type_vec_Auto', 'Vehicle Type_vec_Go Mini', 'Vehicle Type_vec_Go Sedan', 'Vehicle Type_vec_Bike', 'Vehicle Type_vec_Premier Sedan', 'Vehicle Type_vec_eBike', 'Vehicle Type_vec_Uber XL', 'Payment Method_vec_UPI', 'Payment Method_vec_Cash', 'Payment Method_vec_Uber Wallet', 'Payment Method_vec_Credit Card', 'Payment Method_vec_Debit Card', 'Hour_vec_18', 'Hour_vec_17', 'Hour_vec_19', 'Hour_vec_10', 'Hour_vec_16', 'Hour_vec_20', 'Hour_vec_11', 'Hour_vec_21', 'Hour_vec_9', 'Hour_vec_15', 'Hour_vec_14', 'Hour_vec_12', 'Hour_vec_8', 'Hour_vec_13', 'Hour_vec_22', 'Hour_vec_7', 'Hour_vec_6', 'Hour_vec_23', 'Hour_vec_5', 'Hour_vec_0', 'Hour_vec_2', 'Hour_vec_3', 'Hour_vec_4', 'Hour_vec_1', 'DayOfWeek_vec_7', 'DayOfWeek_vec_3', 'DayOfWeek_vec_1', 'DayOfWeek_vec_6', 'DayOfWeek_vec_4', 'DayOfWeek_vec_2', 'DayOfWeek_vec_5', 'Month_vec_1', 'Month_vec_3', 'Month_vec_10', 'Month_vec_7', 'Month_vec_5', 'Month_vec_6', 'Month_vec_12', 'Month_vec_4', 'Month_vec_8', 'Month_vec_11', 'Month_vec_9',

In [35]:
importances = rf_model.featureImportances.toArray()
importance_data = [(feature_names[i], float(importances[i]))for i in range(len(feature_names))]
importance_df = spark.createDataFrame(importance_data,["Feature", "Importance"])

# Sort from highest to lowest importance
importance_df = importance_df.orderBy(col("Importance").desc())

importance_df.show(20, truncate=False)


+------------------------------+--------------------+
|Feature                       |Importance          |
+------------------------------+--------------------+
|Avg VTAT                      |0.1150809131140762  |
|Booking Value                 |0.11226058654551944 |
|Customer Rating               |0.0956754528244088  |
|Ride Distance                 |0.0944531066054288  |
|Avg CTAT                      |0.08743093245931786 |
|DayOfWeek_vec_3               |0.019640554150751915|
|Hour_vec_8                    |0.018855698970600434|
|Hour_vec_13                   |0.01745850993292213 |
|Hour_vec_3                    |0.017145498760657532|
|Hour_vec_1                    |0.016821547777566045|
|Hour_vec_20                   |0.015916770035366356|
|Vehicle Type_vec_Go Mini      |0.01582350283497823 |
|Vehicle Type_vec_Bike         |0.015768647104559042|
|DayOfWeek_vec_2               |0.01393011317084144 |
|Payment Method_vec_UPI        |0.013160186536103203|
|Month_vec_2                

Since it looks like the numerical features have highest importance, another test dataset with only numerical features potentially should be created. However, there is an understanding that for the categorical features, they have been split into further features so their importance may be split over several different features. Therefore below, these split features will be aggreagted together to show the overall importance compared back to the original features.

In [36]:
from pyspark.sql.functions import split

#finding the original features based on the additional name _vec_ used in the above vector assessmbler
importance_grouped = importance_df.withColumn("Original_Feature",
    when(col("Feature").contains("_vec_"),split(col("Feature"), "_vec_").getItem(0)).otherwise(col("Feature")))
#grouping and summing the importance values for the original features
grouped_importance = (importance_grouped.groupBy("Original_Feature").agg(sum("Importance").alias("Total_Importance"))
    .orderBy(col("Total_Importance").desc()))

grouped_importance.show(truncate=False)

+----------------+--------------------+
|Original_Feature|Total_Importance    |
+----------------+--------------------+
|Hour            |0.20054581571751665 |
|Avg VTAT        |0.1150809131140762  |
|Booking Value   |0.11226058654551944 |
|Customer Rating |0.0956754528244088  |
|Ride Distance   |0.0944531066054288  |
|Avg CTAT        |0.08743093245931786 |
|Month           |0.08601662799432805 |
|Vehicle Type    |0.0839229955218349  |
|DayOfWeek       |0.07066377000671546 |
|Payment Method  |0.053949799210853765|
+----------------+--------------------+



Reviewing the importance of difference features, the following have an importance over 9.4%: Hour, Avg VTAT, Booking Value, Customer Rating and Ride Distance. Acknowledging this is only importance shown for a general random forest model, it will be used as a basis for the final dataset and modelling as a decision tree type model makes sense for a ratings type feature. Grouping these numerical values into specific ranges then deciding if that would lead to a specific range of rating would be the best approach for this datatset. Further, these features make sense from a realistic point of view. Customers most important features would be down to price, time it takes for driver to reach the pickup location and time of day. 

In [37]:
final_categoryInputCols = ["Hour"]
final_numericInputCols = ["Avg VTAT", "Booking Value", "Ride Distance", "Customer Rating"]

#completing vector assembly for final category inputs

outputCols2 = [f'{x}_index' for x in final_categoryInputCols]
# Create index values for categorical input features
inputIndexer2 = StringIndexer(inputCols=final_categoryInputCols,outputCols=outputCols2,handleInvalid="keep") 
train_df_indexed_final = inputIndexer2.fit(train_df).transform(train_df)

# Input columns are the indexed categorical input columns
inputCols_OHE2 = outputCols2
outputCols_OHE2 = [f'{x}_vec' for x in final_categoryInputCols]

#Define OneHotEncoder with the appropriate columns
encoder2 = OneHotEncoder(inputCols=inputCols_OHE2,outputCols=outputCols_OHE2)
model2 = encoder2.fit(train_df_indexed_final)
train_df_encoded2 = model2.transform(train_df_indexed_final)

# inputCols are all the encoded columns from OHE plus numerical columns
assemblerInputs2 = outputCols_OHE2 + final_numericInputCols
assembler2 = VectorAssembler(inputCols=assemblerInputs2,outputCol="features")
train_df_final= assembler2.transform(train_df_encoded2)

#number of features within the new dataset, should be 24+4=28
print(len(train_df_final.select("features").first()["features"]))

train_final, test_final = train_df_final.randomSplit([0.7, 0.3], seed = 2020)
print("Final training Dataset Count: " + str(train_final.count()))
print("Final test Dataset Count: " + str(test_final.count()))

28
Final training Dataset Count: 45555
Final test Dataset Count: 19646


In [38]:
train_final.show(10)

+----------------+--------------+----------------+-------------+----------------+----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+----+---------+-----+----------+---------------+--------------------+
|      Booking ID|Booking Status|     Customer ID| Vehicle Type| Pickup Location|   Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|Hour|DayOfWeek|Month|Hour_index|       Hour_vec|            features|
+----------------+--------------+----------------+-------------+----------------+----------------+--------+--------+---------------------------+------------------------

In [39]:
#create three seperate general models with final dataset to investigate metrics

lr_final = LinearRegression(featuresCol = "features", labelCol=numericOutputCol)
lrModel_final = lr.fit(train_final)
lrPredictions_final = lrModel_final.transform(test_final)

print("linear regression")
lr_results_final = evaluate_regression(lrPredictions_final,numericOutputCol)

dt_final = DecisionTreeRegressor(featuresCol="features",labelCol=numericOutputCol,seed=42)
dt_model_final = dt_final.fit(train_final)
dt_predictions_final = dt_model_final.transform(test_final)

print("decision tree")
dt_results_final = evaluate_regression(dt_predictions_final,numericOutputCol)

rf_final = RandomForestRegressor(featuresCol="features",labelCol=numericOutputCol,seed=42)
rf_model_final = rf.fit(train_final)
rf_predictions_final = rf_model_final.transform(test_final)

print("random forest")
rf_results_final = evaluate_regression(rf_predictions_final,numericOutputCol)

26/10/03 09:49:24 WARN Instrumentation: [dc959905] regParam is zero, which might cause numerical instability and overfitting.
26/10/03 09:49:25 WARN Instrumentation: [dc959905] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


linear regression
RMSE: 0.438
R²:   -0.000
MAE:   0.331
decision tree
RMSE: 0.439
R²:   -0.006
MAE:   0.333
random forest
RMSE: 0.438
R²:   -0.001
MAE:   0.331


In [40]:
#regression parameter investigate with linear model
lr_final_l2 = LinearRegression(featuresCol="features",labelCol=numericOutputCol,regParam=0.01,elasticNetParam=0.0)
lr_final_l2_model = lr_final_l2.fit(train_final)
lr_final_l2_Predictions = lr_final_l2_model.transform(test_final)
lr_l2_results_final = evaluate_regression(lr_final_l2_Predictions,numericOutputCol)

RMSE: 0.438
R²:   -0.000
MAE:   0.331


In [41]:
#decision tree with specific depth and node values
dt_final_1 = DecisionTreeRegressor(featuresCol="features",labelCol=numericOutputCol,maxDepth=20,minInstancesPerNode=3,seed=42)
dt_final_1_model = dt_final_1.fit(train_final)
dt_final_1_predictions = dt_final_1_model.transform(test_final)
dt_final_1_results = evaluate_regression(dt_final_1_predictions,numericOutputCol)

26/10/03 09:49:32 WARN DAGScheduler: Broadcasting large task binary with size 1130.7 KiB
26/10/03 09:49:32 WARN DAGScheduler: Broadcasting large task binary with size 1310.6 KiB
26/10/03 09:49:32 WARN DAGScheduler: Broadcasting large task binary with size 1504.5 KiB


RMSE: 0.499
R²:   -0.303
MAE:   0.385


In [42]:
#random forest with specific numebr of trees, depth and nodes
rf_final_1 = RandomForestRegressor(featuresCol="features",labelCol=numericOutputCol,numTrees=100,maxDepth=15,minInstancesPerNode=8,
    featureSubsetStrategy="sqrt",seed=42)
rf_final_1_model = rf_final_1.fit(train_final)
rf_final_1_predictions = rf_final_1_model.transform(test_final)
rf_final_1_results = evaluate_regression(rf_final_1_predictions,numericOutputCol)

26/10/03 09:49:35 WARN DAGScheduler: Broadcasting large task binary with size 1024.1 KiB
26/10/03 09:49:35 WARN DAGScheduler: Broadcasting large task binary with size 1433.5 KiB
26/10/03 09:49:35 WARN DAGScheduler: Broadcasting large task binary with size 1955.3 KiB
26/10/03 09:49:36 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/10/03 09:49:36 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/10/03 09:49:37 WARN DAGScheduler: Broadcasting large task binary with size 4.3 MiB
26/10/03 09:49:38 WARN DAGScheduler: Broadcasting large task binary with size 5.4 MiB
26/10/03 09:49:39 WARN DAGScheduler: Broadcasting large task binary with size 6.7 MiB
26/10/03 09:49:40 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB


RMSE: 0.438
R²:   -0.002
MAE:   0.332


#### 3/4c. Final conclusions on dataset

From the above models, it seems that reducing the dataset further didn't have any impact on the regression statistics. Based on this, reviewing the variance of Driver Ratings and the correlation to the numerical values is done to evaluate if the a ideal model could even be created from this dataset. Based on the values below, the Driver Ratings do have a good distribution but only between 3-5. The standard deviation of 0.436 matches the overal RMSE of the various models above. Looking at the correlation for just the numerical inputs, there is basically no relation between any of these values and the Driver Rating. Similarly for the Hour, the average rating for each hour is basically the same, which highlights that summing importance for a feature seperated into 24 features isn't the best approach. Based on this investigation, it can be concluded that the data available cannot create a model that will predict the Driver Rating. This may highlight that additional factors that aren't recorded may be needed. For example, cleaniness of vechile or communication with driver may be more impactful on Driver Rating, which is a very human metric and could be greatly influenced by many things. 

In [43]:
train_final.groupBy(numericOutputCol).count().orderBy(numericOutputCol).show()

+--------------+-----+
|Driver Ratings|count|
+--------------+-----+
|           3.0|  349|
|           3.1|  729|
|           3.2|  759|
|           3.3|  727|
|           3.4|  752|
|           3.5|  334|
|           3.6|  971|
|           3.7| 1841|
|           3.8| 1873|
|           3.9| 1908|
|           4.0|  972|
|           4.1| 3449|
|           4.2| 6769|
|           4.3| 6943|
|           4.4| 3483|
|           4.5| 2206|
|           4.6| 4615|
|           4.7| 2306|
|           4.8| 1147|
|           4.9| 2262|
+--------------+-----+
only showing top 20 rows


In [44]:
train_final.select(numericOutputCol).describe().show()

+-------+------------------+
|summary|    Driver Ratings|
+-------+------------------+
|  count|             45555|
|   mean|4.2311250067505615|
| stddev|0.4363564461287325|
|    min|               3.0|
|    max|               5.0|
+-------+------------------+



In [45]:
numericInputCols = ["Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance", "Customer Rating"]

for feature in numericInputCols:
    
    correlation = train_df_final.stat.corr(feature,"Driver Ratings")
    
    print(f"{feature}: {correlation:.4f}")

Avg VTAT: -0.0027
Avg CTAT: 0.0055
Booking Value: 0.0015
Ride Distance: -0.0016
Customer Rating: 0.0019


In [46]:
from pyspark.sql.functions import avg, count, round

train_df.groupBy("Hour").agg(
    round(
        avg("Driver Ratings"),
        3
    ).alias("Mean_Driver_Rating"),
    
    count("*").alias("Count")
).orderBy("Hour").show(24)

+----+------------------+-----+
|Hour|Mean_Driver_Rating|Count|
+----+------------------+-----+
|   0|             4.258|  611|
|   1|             4.208|  581|
|  10|             4.224| 4263|
|  11|             4.227| 3664|
|  12|              4.24| 3017|
|  13|             4.221| 2362|
|  14|             4.228| 3032|
|  15|             4.233| 3527|
|  16|             4.228| 4190|
|  17|             4.231| 4823|
|  18|             4.221| 5362|
|  19|             4.232| 4786|
|   2|             4.234|  603|
|  20|             4.248| 4188|
|  21|             4.235| 3585|
|  22|             4.237| 2361|
|  23|             4.225| 1227|
|   3|             4.218|  593|
|   4|             4.241|  586|
|   5|             4.246| 1189|
|   6|             4.226| 1833|
|   7|             4.232| 2337|
|   8|             4.237| 2921|
|   9|             4.235| 3560|
+----+------------------+-----+



#### 3/4d. Model Evaluation Table

The final evaluation table will be created from the six regression models created using the 28 feature dataset, even though it was concluded none of these models are ideal based on the data available. Based on the below metrics, the model chosen for Part B investigation is 'Decision Tree Default'. 

In [47]:
import builtins
model_results = [
    ("Linear Regression",builtins.round(lr_results_final[0],3),builtins.round(lr_results_final[1],3),builtins.round(lr_results_final[2],3)),
    ("Linear Regression L2 Regression",builtins.round(lr_l2_results_final[0],3),builtins.round(lr_l2_results_final[1],3),builtins.round(lr_l2_results_final[2],3)),
    ("Decision Tree Default",builtins.round(dt_results_final[0],3),builtins.round(dt_results_final[1],3),builtins.round(dt_results_final[2],3)),
    ("Decision Tree D-20,N-3",builtins.round(dt_final_1_results[0],3),builtins.round(dt_final_1_results[1],3),builtins.round(dt_final_1_results[2],3)),
    ("Random Forest Default",builtins.round(rf_results_final[0],3),builtins.round(rf_results_final[1],3),builtins.round(rf_results_final[2],3)),
    ("Random Forest T-100,D-15,N-8",builtins.round(rf_final_1_results[0],3),builtins.round(rf_final_1_results[1],3),builtins.round(rf_final_1_results[2],3)),
]

results_df = spark.createDataFrame(model_results,["Model", "RMSE", "R2", "MAE"])

results_df.show(truncate=False)

+-------------------------------+-----+------+-----+
|Model                          |RMSE |R2    |MAE  |
+-------------------------------+-----+------+-----+
|Linear Regression              |0.438|-0.0  |0.331|
|Linear Regression L2 Regression|0.438|-0.0  |0.331|
|Decision Tree Default          |0.439|-0.006|0.333|
|Decision Tree D-20,N-3         |0.499|-0.303|0.385|
|Random Forest Default          |0.438|-0.001|0.331|
|Random Forest T-100,D-15,N-8   |0.438|-0.002|0.332|
+-------------------------------+-----+------+-----+



### 5. Model Persistence

In [ ]:
#saving best model 
dt_model_final.write().overwrite().save("models/dt_model_final")

In [49]:
#dedicated notebook cell reloading saved model and confirming the schema matches
from pyspark.ml.regression import DecisionTreeRegressionModel

dt_model_loaded = DecisionTreeRegressionModel.load("models/dt_model_final")

#sample of test dataset
sample = test_final.limit(10)

# Predictions from original and loaded model
original_predictions = dt_model_final.transform(sample)
loaded_predictions = dt_model_loaded.transform(sample)

# Compare schemas
print("Schemas match:",original_predictions.schema == loaded_predictions.schema)

Schemas match: True


### Part B - Streaming, Kafka and reflection

Part B has four components: a Kafka producer, a Spark Structured Steaming consumer, a windowed agreegation and an embedded reflection

#### 1. Streaming data simulation
You will simulate a real-time data stream by replaying records from the held-out streaming subset through Kafta. Each batch represents a group of newly arriving events. The Spark consumer ingests these events, applies the persisted ML model and generates predictions in near real-time. The objective is to demonstrate streaming data processing, not collect live data from an external system.

In [52]:
#first create pipelinemodel that can be used to complete the processessing in part a for streamed data
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Final model inputs
final_categoryInputCols = ["Hour"]
final_numericInputCols = ["Avg VTAT","Booking Value", "Ride Distance","Customer Rating"]

# Stage 1: Index categorical variable
outputCols2 = [f"{x}_index"for x in final_categoryInputCols]
inputIndexer2 = StringIndexer(inputCols=final_categoryInputCols,outputCols=outputCols2,handleInvalid="keep")

# Stage 2: One-hot encode indexed variable
inputCols_OHE2 = outputCols2
outputCols_OHE2 = [f"{x}_vec"for x in final_categoryInputCols]
encoder2 = OneHotEncoder(inputCols=inputCols_OHE2,outputCols=outputCols_OHE2)

# Stage 3: Assemble model features
assemblerInputs2 = (outputCols_OHE2+ final_numericInputCols)
assembler2 = VectorAssembler(inputCols=assemblerInputs2,outputCol="features",handleInvalid="keep")

# Build pipeline
stages = [inputIndexer2,encoder2,assembler2]

pipeline = Pipeline(stages=stages)

pipelineModel = pipeline.fit(train_df)
train_df_final = pipelineModel.transform(train_df)


In [53]:
#saving pipeline model so it can be called later
pipelineModel.write() .overwrite().save("models/preprocessing_pipeline")

26/10/03 09:51:26 ERROR FileOutputCommitter: Mkdirs failed to create file:/usr/src/app/models/preprocessing_pipeline/metadata/_temporary/0
26/10/03 09:51:26 ERROR Utils: Aborting task
java.io.IOException: Mkdirs failed to create file:/usr/src/app/models/preprocessing_pipeline/metadata/_temporary/0/_temporary/attempt_202610030951266561624337805069195_0583_m_000000_2644 (exists=false, cwd=file:/usr/src/app)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:774)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:759)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1234)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1211)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1092)
	at org.apache.spark.sql.execution.datasources.CodecStreams$.createOutputStream(CodecStreams.scala:72)
	at org.apache.spark.sql.execution.datasources.text.TextOutputWriter.<init>(TextOutputWriter.scala:33)
	at org.apache.spark.sql.execu

Py4JJavaError: An error occurred while calling o10511.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 583.0 failed 1 times, most recent failure: Lost task 0.0 in stage 583.0 (TID 2644) (26ae54ea911b executor driver): java.io.IOException: Mkdirs failed to create file:/usr/src/app/models/preprocessing_pipeline/metadata/_temporary/0/_temporary/attempt_202610030951266561624337805069195_0583_m_000000_2644 (exists=false, cwd=file:/usr/src/app)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:774)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:759)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1234)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1211)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1092)
	at org.apache.spark.sql.execution.datasources.CodecStreams$.createOutputStream(CodecStreams.scala:72)
	at org.apache.spark.sql.execution.datasources.text.TextO

In [ ]:
batch_df = (
    batch_df
    .withColumn("Avg VTAT", col("Avg VTAT").cast("float"))
    .withColumn("Booking Value", col("Booking Value").cast("float"))
    .withColumn("Ride Distance", col("Ride Distance").cast("float"))
    .withColumn("Customer Rating", col("Customer Rating").cast("float"))
    .withColumn("Hour", hour("Time").cast("string"))
)

processed_df = pipelineModel.transform(batch_df)

predictions = dt_model.transform(processed_df)
